In [1]:
#pip install -q -U \ langchain \ langchain-core \ langchain-community \ langchain-google-genai \ faiss-cpu

In [2]:
2+2

4

In [3]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from dotenv import load_dotenv

C:\Users\vaish\AppData\Local\Temp\ipykernel_5260\3149061991.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
load_dotenv()

api_key=os.getenv('GEMINI_API_KEY')


In [5]:
#2.Initialize the embedding model

embedder=GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)



In [6]:
#3 Initialize chat model

llm=ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.0

)

In [ ]:
#4.Mount the faiss

DB_DIR="faiss_extract_once_db"

vectorstore=FAISS.load_local(
    folder_path=DB_DIR,
    embeddings=embedder,
    allow_dangerous_deserialization=True
)

In [8]:
#5.Convert the database inot langchain retriever

retriever=vectorstore.as_retriever(search_kwags={"k":3})

In [ ]:
#6 The context formatter

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    

In [11]:
#7. Define the RAG Mega-Prompt

template = """

You are an elite MLOps auditing algorithm for the 'Extract Once, Judge Many' project.
Your core directive is to answer the user's query using STRICTLY the context provided below.

CRITICAL RULES:

1. Anti-Hallucination Protocol: If the answer is not explicitly contained in the context, you must output: "I cannot answer this based on the provided corporate documents." Do not attempt to guess.
2. Do not use your pre-trained internet knowledge.
3. Keep your answer professional, highly concise, and cite the author if available.

====================

RETRIEVED CONTEXT:

{context}

====================

USER QUERY: {question}

FINAL ANSWER:

"""

rag_prompt=PromptTemplate.from_template(template)

In [20]:
#8.THE LCEL CHAIN--langchain expression languague

#retriever|prompt|llm|parser

rag_chain=(
    {"context":retriever | format_docs,"question":RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

#takes the query grabs the users query and passes 

In [21]:
#9. Execute the valid query

user_question="What kind of architecture are we talking about?"

final_response=rag_chain.invoke(user_question)
print(f"Ai response is {final_response}")

Ai response is FINAL ANSWER:

Tony has proposed moving to a distributed architecture.


C:\Users\vaish\AppData\Local\Temp\ipykernel_5260\77622800.py:5: RuntimeWarning: coroutine 'RunnableSequence.ainvoke' was never awaited
  final_response=rag_chain.invoke(user_question)


In [23]:
user_question="what is newton school?"

final_response=rag_chain.invoke(user_question)
print(f"Ai response is {final_response}")

Ai response is I cannot answer this based on the provided corporate documents.
